In [1]:
import time
import pandas as pd
from zhinst.toolkit import Session
import numpy as np
import matplotlib.pyplot as plt
from sagnalysis import get_noise_data
import os
# import matplotlib
# matplotlib.use('TkAgg')


session = Session("localhost", hf2=True)
device = session.connect_device("DEV1004")

In [36]:
%matplotlib notebook


fixed_time_constant = 10e-3
filename = "ShotNoiseQ_EOMoff_1.csv"
# filename = "ShotNoiseQ_1plus.csv"


while True:
    try:
        light_intensity = float(input("Enter the light intensity, in nW: "))
    except ValueError:
        print()
        cont = input("Invalid input. Do you want to enter another light intensity? (y/n): ").strip().lower()
        if cont != 'y':
            break
        continue

    # Collect data
    data = get_noise_data(device, TcSet=fixed_time_constant, numSamples=300, demod=3, tmeasure=60)
    # should take 10 seconds given this time constant

    # Process the results
    TcSet = data.TcSet.mean() 
    TcReal = data.TcReal.mean() 
    y1 = data.y.mean()
    x1 = data.x.mean()
    dx1 =data.x.std()
    dy1 =data.y.std()
    numMeas = data.timestamp.count()

    df = pd.DataFrame({
        'LightIntensity': [light_intensity],
        'TcSet': [TcSet],
        'TcReal': [TcReal],
        'x1': [x1],
        'y1': [y1],
        'dx1': [dx1],
        'dy1': [dy1],
        'numMeas': [numMeas]
    })
    df["xSensitivity"] = df.dx1 * np.sqrt(df.TcReal)
    df["ySensitivity"] = df.dy1 * np.sqrt(df.TcReal)
    df["xFracErr"] = df.dx1 / df.x1
    df["yFracErr"] = df.dy1 / df.y1

    df['localTime'] = time.strftime('%Y-%m-%d %H:%M:%S')
    df['time'] = time.time()
    df.to_csv(filename, mode='a', header=not os.path.exists(filename), index=False)

    # # Plot the file sensitivities as a function of light intensity
    # plotable = pd.read_csv(filename)
    # plt.figure()
    # plt.plot(plotable.LightIntensity, plotable.xSensitivity, 'o', label='X Sensitivity'
    #          , color='blue')
    # plt.plot(plotable.LightIntensity, plotable.ySensitivity, 'o', label='Y Sensitivity'
    #             , color='red')
    # plt.xlabel('Light Intensity (nW)')
    # plt.ylabel('Sensitivity (V/sqrtHz)')
    # plt.legend()

    print(df)

   LightIntensity  TcSet    TcReal            x1            y1       dx1  \
0          1283.0   0.01  0.010164  2.354087e-07 -3.089290e-07  0.000005   

        dy1  numMeas  xSensitivity  ySensitivity   xFracErr   yFracErr  \
0  0.000005      300  4.875887e-07  5.092444e-07  20.544151 -16.350262   

             localTime          time  
0  2024-06-24 23:17:24  1.719285e+09  
   LightIntensity  TcSet    TcReal            x1            y1       dx1  \
0          1283.0   0.01  0.010164 -3.489899e-07  4.757022e-08  0.000005   

        dy1  numMeas  xSensitivity  ySensitivity   xFracErr   yFracErr  \
0  0.000005      300  4.693805e-07  4.767144e-07 -13.340412  99.398577   

             localTime          time  
0  2024-06-24 23:18:10  1.719285e+09  

